In [1]:
import pandas as pd
import seaborn as sns

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [2]:
df_jan = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
df_feb = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')

# Q1

In [3]:
print(len(df_jan.columns))

19


# Q2

In [4]:
df_jan.columns
df_jan.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [5]:
def duration_feature(df):
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    
    df['duration'] = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
    df['duration'] = df.duration.apply(lambda td: td.total_seconds() / 60)
    return df

In [6]:
df_jan = duration_feature(df_jan)
df_feb = duration_feature(df_feb)

In [7]:
df_jan['duration'].std()

np.float64(42.59435124195457)

# Q3

In [8]:
def remove_col_outliers(df, upper_bound, lower_bound, col_name):
    df = df[df[col_name] >= lower_bound]
    df = df[df[col_name] <= upper_bound]
    return df

In [9]:
rows_with_outliers, _ = df_jan.shape
print(rows_with_outliers)

3066766


In [10]:
df_jan = remove_col_outliers(df_jan, 60, 1, 'duration')
df_feb = remove_col_outliers(df_feb, 60, 1, 'duration')

In [11]:
rows_without_outliers, _ = df_jan.shape
print(rows_without_outliers)

3009173


In [12]:
print(int((rows_without_outliers / rows_with_outliers) * 100))

98


# Q4

In [13]:
dv = DictVectorizer()

features = ['PULocationID','DOLocationID']
df_jan[features] = df_jan[features].astype(str)
df_feb[features] = df_feb[features].astype(str)

jan_dicts = df_jan[features].to_dict(orient='records')
X_train = dv.fit_transform(jan_dicts)

feb_dicts = df_feb[features].to_dict(orient='records')
X_eval = dv.transform(feb_dicts)

In [14]:
print(X_train.shape)

(3009173, 515)


# Q5

In [15]:
target = 'duration'
y_train = df_jan[target].values
y_eval = df_feb[target].values

In [16]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)
root_mean_squared_error(y_train, y_pred)

7.649261932456945

# Q6

In [17]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_eval)
root_mean_squared_error(y_eval, y_pred)

7.811818758819308